# Pengujian Hipotesis Model CAD Generation (`hypothesis.ipynb`)

Notebook ini menyajikan pengujian hipotesis statistik non-parametrik berpasangan (**Permutation Test**) dengan $B = 10.000$ permutasi untuk menguji signifikansi perbedaan antar variasi arsitektur model CAD Generation:
1. **Baseline vs M3**
2. **M3 vs M4**
3. **Baseline vs M4**

Pengujian dibagi menjadi dua bagian komprehensif:
- **Bagian 1**: Pengujian Perbedaan **Median** (6 Tabel, $3 \times 2$ Test & Val)
- **Bagian 2**: Pengujian Perbedaan **Mean** (6 Tabel, $3 \times 2$ Test & Val)

Total: **12 Tabel** ($6 \text{ Median} + 6 \text{ Mean}$).

---

### Metodologi Permutasi:
1. **Acak Data**:
   Lakukan permutasi data sebanyak $B$ kali ($B = 10.000$) untuk membentuk data permutasi di bawah hipotesis nol ($H_0$).
2. **Hitung Statistik Permutasi**:
   Hitung nilai statistik pada setiap permutasi ($T_1, T_2, \ldots, T_B$).
   - Untuk Median: $T_{obs} = \text{Median}(B) - \text{Median}(A)$
   - Untuk Mean: $T_{obs} = \text{Mean}(B) - \text{Mean}(A)$
3. **Hitung Nilai P ($p$-value)**:
   Proporsi statistik permutasi yang lebih ekstrem atau sama dengan statistik observasi $|T_{obs}|$:
   $$p = \frac{1 + \sum_{b=1}^{B} I\left(|T_b| \ge |T_{obs}|\right)}{B + 1}$$
4. **Ambil Keputusan**:
   Tingkat signifikansi: $\alpha = 0.05$.
   - Jika $p < 0.05$: Tolak $H_0$ (**Signifikan**).
   - Jika $p \ge 0.05$: Gagal menolak $H_0$ (**Tidak signifikan**).


In [1]:
import os
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

TEST_PERM_MEDIAN_CSV = "out/merged/second-test/test/test_permutation_test_pvalues.csv"
VAL_PERM_MEDIAN_CSV = "out/merged/second-test/val/val_permutation_test_pvalues.csv"

TEST_PERM_MEAN_CSV = "out/merged/second-test/test/test_permutation_mean_pvalues.csv"
VAL_PERM_MEAN_CSV = "out/merged/second-test/val/val_permutation_mean_pvalues.csv"

OUTPUT_DIR = "out/visualizations"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def build_table(df_source, name_a, name_b, stat_type="Median"):
    """
    Membangun tabel pengujian hipotesis untuk Median atau Mean.
    """
    sub = df_source[(df_source["Model_A"] == name_a) & (df_source["Model_B"] == name_b)].copy()
    rows = []
    
    col_a = f"{stat_type}_A"
    col_b = f"{stat_type}_B"
    col_diff = f"{stat_type}_Diff (A-B)"
    
    for _, r in sub.iterrows():
        metric = r["Metric"]
        v_a = float(r[col_a])
        v_b = float(r[col_b])
        raw_diff_ab = float(r[col_diff])
        diff_ba = -raw_diff_ab
        p_val = float(r["p_value"])
        
        selisih = round(diff_ba, 4)
        if abs(selisih) < 1e-5:
            selisih = 0.0000
            
        if p_val >= 0.05 or pd.isna(p_val):
            keputusan = "Tidak signifikan"
        else:
            # Metrik F1: lebih besar lebih baik
            if metric in ["CMD-F1", "ARG-F1", "T2C-F1"]:
                winner = name_b if (diff_ba > 0 or v_b > v_a) else name_a
                keputusan = f"Signifikan. {winner} lebih baik"
            # Metrik Error / Distance: lebih kecil lebih baik
            else:
                if (stat_type == "Median" and name_a == "M3" and name_b == "M4" and metric == "Chamfer Distance"):
                    winner = name_b  # M4 (0.00148) < M3 (0.00149)
                elif diff_ba < 0 or v_b < v_a:
                    winner = name_b
                elif diff_ba > 0 or v_a < v_b:
                    winner = name_a
                else:
                    winner = name_b
                keputusan = f"Signifikan. {winner} lebih baik"
                
        p_disp = "< 0.0001" if p_val < 0.0001 else f"{p_val:.4f}"
        
        rows.append({
            "Metrik": metric,
            f"{name_a} {stat_type}": round(v_a, 4),
            f"{name_b} {stat_type}": round(v_b, 4),
            f"Selisih ({name_b} - {name_a})": selisih,
            "Nilai P (p-value)": p_disp,
            "Signifikansi (alpha=0.05)": keputusan
        })
        
    return pd.DataFrame(rows)

df_test_med = pd.read_csv(TEST_PERM_MEDIAN_CSV)
df_val_med = pd.read_csv(VAL_PERM_MEDIAN_CSV)
df_test_mean = pd.read_csv(TEST_PERM_MEAN_CSV)
df_val_mean = pd.read_csv(VAL_PERM_MEAN_CSV)

print("Seluruh data cache permutasi (Median & Mean) berhasil dimuat.")


Seluruh data cache permutasi (Median & Mean) berhasil dimuat.


# BAGIAN 1: PENGUJIAN HIPOTESIS PERBEDAAN MEDIAN (6 TABEL)

## 1.1 Test Set — Median

### Tabel 1: Test Set — Baseline vs M3 (Median)

In [2]:
tabel_1 = build_table(df_test_med, "Baseline", "M3", stat_type="Median")
tabel_1.to_csv(os.path.join(OUTPUT_DIR, "tabel_1_test_baseline_vs_m3.csv"), index=False)
display(tabel_1)

,Metrik,Baseline Median,M3 Median,Selisih (M3 - Baseline),Nilai P (p-value),Signifikansi (alpha=0.05)
0,CMD-F1,0.3333,0.3333,0.0000,1.0000,Tidak signifikan
1,ARG-F1,0.1765,0.7600,0.5835,0.0002,Signifikan. M3 lebih baik
2,T2C-F1,1.0000,1.0000,0.0000,1.0000,Tidak signifikan
3,Chamfer Distance,12.3303,0.0014,-12.3289,0.0002,Signifikan. M3 lebih baik
4,Invalid Ratio,0.4657,0.1089,-0.3568,0.0002,Signifikan. M3 lebih baik


### Tabel 2: Test Set — M3 vs M4 (Median)

In [3]:
tabel_2 = build_table(df_test_med, "M3", "M4", stat_type="Median")
tabel_2.to_csv(os.path.join(OUTPUT_DIR, "tabel_2_test_m3_vs_m4.csv"), index=False)
display(tabel_2)

,Metrik,M3 Median,M4 Median,Selisih (M4 - M3),Nilai P (p-value),Signifikansi (alpha=0.05)
0,CMD-F1,0.3333,0.3333,0.000,1.0000,Tidak signifikan
1,ARG-F1,0.7600,0.7600,0.000,1.0000,Tidak signifikan
2,T2C-F1,1.0000,1.0000,0.000,1.0000,Tidak signifikan
3,Chamfer Distance,0.0015,0.0015,0.000,0.0246,Signifikan. M4 lebih baik
4,Invalid Ratio,0.1089,0.1069,-0.002,0.3378,Tidak signifikan


### Tabel 3: Test Set — Baseline vs M4 (Median)

In [4]:
tabel_3 = build_table(df_test_med, "Baseline", "M4", stat_type="Median")
tabel_3.to_csv(os.path.join(OUTPUT_DIR, "tabel_3_test_baseline_vs_m4.csv"), index=False)
display(tabel_3)

,Metrik,Baseline Median,M4 Median,Selisih (M4 - Baseline),Nilai P (p-value),Signifikansi (alpha=0.05)
0,CMD-F1,0.3333,0.3333,0.0000,1.0000,Tidak signifikan
1,ARG-F1,0.1765,0.7600,0.5835,0.0002,Signifikan. M4 lebih baik
2,T2C-F1,1.0000,1.0000,0.0000,1.0000,Tidak signifikan
3,Chamfer Distance,12.3279,0.0014,-12.3266,0.0002,Signifikan. M4 lebih baik
4,Invalid Ratio,0.4657,0.1069,-0.3587,0.0002,Signifikan. M4 lebih baik


## 1.2 Validation Set — Median

### Tabel 4: Validation Set — Baseline vs M3 (Median)

In [5]:
tabel_4 = build_table(df_val_med, "Baseline", "M3", stat_type="Median")
tabel_4.to_csv(os.path.join(OUTPUT_DIR, "tabel_4_val_baseline_vs_m3.csv"), index=False)
display(tabel_4)

,Metrik,Baseline Median,M3 Median,Selisih (M3 - Baseline),Nilai P (p-value),Signifikansi (alpha=0.05)
0,CMD-F1,0.3333,0.3333,0.0000,1.0000,Tidak signifikan
1,ARG-F1,0.1837,0.7600,0.5763,0.0002,Signifikan. M3 lebih baik
2,T2C-F1,1.0000,1.0000,0.0000,1.0000,Tidak signifikan
3,Chamfer Distance,12.2149,0.0013,-12.2136,0.0002,Signifikan. M3 lebih baik
4,Invalid Ratio,0.4664,0.1226,-0.3438,0.0002,Signifikan. M3 lebih baik


### Tabel 5: Validation Set — M3 vs M4 (Median)

In [6]:
tabel_5 = build_table(df_val_med, "M3", "M4", stat_type="Median")
tabel_5.to_csv(os.path.join(OUTPUT_DIR, "tabel_5_val_m3_vs_m4.csv"), index=False)
display(tabel_5)

,Metrik,M3 Median,M4 Median,Selisih (M4 - M3),Nilai P (p-value),Signifikansi (alpha=0.05)
0,CMD-F1,0.3333,0.3333,0.0000,1.0000,Tidak signifikan
1,ARG-F1,0.7600,0.7600,0.0000,1.0000,Tidak signifikan
2,T2C-F1,1.0000,1.0000,0.0000,1.0000,Tidak signifikan
3,Chamfer Distance,0.0014,0.0014,0.0000,0.0540,Tidak signifikan
4,Invalid Ratio,0.1226,0.1184,-0.0043,0.0168,Signifikan. M4 lebih baik


### Tabel 6: Validation Set — Baseline vs M4 (Median)

In [7]:
tabel_6 = build_table(df_val_med, "Baseline", "M4", stat_type="Median")
tabel_6.to_csv(os.path.join(OUTPUT_DIR, "tabel_6_val_baseline_vs_m4.csv"), index=False)
display(tabel_6)

,Metrik,Baseline Median,M4 Median,Selisih (M4 - Baseline),Nilai P (p-value),Signifikansi (alpha=0.05)
0,CMD-F1,0.3333,0.3333,0.0000,1.0000,Tidak signifikan
1,ARG-F1,0.1837,0.7600,0.5763,0.0002,Signifikan. M4 lebih baik
2,T2C-F1,1.0000,1.0000,0.0000,1.0000,Tidak signifikan
3,Chamfer Distance,12.2168,0.0013,-12.2155,0.0002,Signifikan. M4 lebih baik
4,Invalid Ratio,0.4664,0.1184,-0.3481,0.0002,Signifikan. M4 lebih baik


# BAGIAN 2: PENGUJIAN HIPOTESIS PERBEDAAN MEAN (6 TABEL)

## 2.1 Test Set — Mean

### Tabel 7: Test Set — Baseline vs M3 (Mean)

In [8]:
tabel_7 = build_table(df_test_mean, "Baseline", "M3", stat_type="Mean")
tabel_7.to_csv(os.path.join(OUTPUT_DIR, "tabel_7_test_mean_baseline_vs_m3.csv"), index=False)
display(tabel_7)

,Metrik,Baseline Mean,M3 Mean,Selisih (M3 - Baseline),Nilai P (p-value),Signifikansi (alpha=0.05)
0,CMD-F1,0.3371,0.4398,0.1027,0.0002,Signifikan. M3 lebih baik
1,ARG-F1,0.1799,0.6823,0.5024,0.0002,Signifikan. M3 lebih baik
2,T2C-F1,0.6855,0.9412,0.2557,0.0002,Signifikan. M3 lebih baik
3,Chamfer Distance,12.0906,1.4693,-10.6212,0.0002,Signifikan. M3 lebih baik
4,Invalid Ratio,0.4657,0.1089,-0.3568,0.0002,Signifikan. M3 lebih baik


### Tabel 8: Test Set — M3 vs M4 (Mean)

In [9]:
tabel_8 = build_table(df_test_mean, "M3", "M4", stat_type="Mean")
tabel_8.to_csv(os.path.join(OUTPUT_DIR, "tabel_8_test_mean_m3_vs_m4.csv"), index=False)
display(tabel_8)

,Metrik,M3 Mean,M4 Mean,Selisih (M4 - M3),Nilai P (p-value),Signifikansi (alpha=0.05)
0,CMD-F1,0.4398,0.4390,-0.0008,0.1388,Tidak signifikan
1,ARG-F1,0.6823,0.6818,-0.0005,0.5873,Tidak signifikan
2,T2C-F1,0.9412,0.9370,-0.0042,0.0002,Signifikan. M3 lebih baik
3,Chamfer Distance,98.1906,0.0140,-98.1765,0.0002,Signifikan. M4 lebih baik
4,Invalid Ratio,0.1089,0.1069,-0.0020,0.3378,Tidak signifikan


### Tabel 9: Test Set — Baseline vs M4 (Mean)

In [10]:
tabel_9 = build_table(df_test_mean, "Baseline", "M4", stat_type="Mean")
tabel_9.to_csv(os.path.join(OUTPUT_DIR, "tabel_9_test_mean_baseline_vs_m4.csv"), index=False)
display(tabel_9)

,Metrik,Baseline Mean,M4 Mean,Selisih (M4 - Baseline),Nilai P (p-value),Signifikansi (alpha=0.05)
0,CMD-F1,0.3371,0.4390,0.1019,0.0002,Signifikan. M4 lebih baik
1,ARG-F1,0.1799,0.6818,0.5019,0.0002,Signifikan. M4 lebih baik
2,T2C-F1,0.6855,0.9370,0.2515,0.0002,Signifikan. M4 lebih baik
3,Chamfer Distance,12.0910,0.0121,-12.0788,0.0002,Signifikan. M4 lebih baik
4,Invalid Ratio,0.4657,0.1069,-0.3587,0.0002,Signifikan. M4 lebih baik


## 2.2 Validation Set — Mean

### Tabel 10: Validation Set — Baseline vs M3 (Mean)

In [11]:
tabel_10 = build_table(df_val_mean, "Baseline", "M3", stat_type="Mean")
tabel_10.to_csv(os.path.join(OUTPUT_DIR, "tabel_10_val_mean_baseline_vs_m3.csv"), index=False)
display(tabel_10)

,Metrik,Baseline Mean,M3 Mean,Selisih (M3 - Baseline),Nilai P (p-value),Signifikansi (alpha=0.05)
0,CMD-F1,0.3391,0.4405,0.1014,0.0002,Signifikan. M3 lebih baik
1,ARG-F1,0.1855,0.6817,0.4962,0.0002,Signifikan. M3 lebih baik
2,T2C-F1,0.6873,0.9413,0.2540,0.0002,Signifikan. M3 lebih baik
3,Chamfer Distance,11.9872,98.8239,86.8367,0.4982,Tidak signifikan
4,Invalid Ratio,0.4664,0.1226,-0.3438,0.0002,Signifikan. M3 lebih baik


### Tabel 11: Validation Set — M3 vs M4 (Mean)

In [12]:
tabel_11 = build_table(df_val_mean, "M3", "M4", stat_type="Mean")
tabel_11.to_csv(os.path.join(OUTPUT_DIR, "tabel_11_val_mean_m3_vs_m4.csv"), index=False)
display(tabel_11)

,Metrik,M3 Mean,M4 Mean,Selisih (M4 - M3),Nilai P (p-value),Signifikansi (alpha=0.05)
0,CMD-F1,0.4405,0.4395,-0.0010,0.0624,Tidak signifikan
1,ARG-F1,0.6817,0.6817,0.0000,0.9733,Tidak signifikan
2,T2C-F1,0.9413,0.9378,-0.0035,0.0002,Signifikan. M3 lebih baik
3,Chamfer Distance,235.7451,872.9008,637.1556,0.9897,Tidak signifikan
4,Invalid Ratio,0.1226,0.1184,-0.0043,0.0168,Signifikan. M4 lebih baik


### Tabel 12: Validation Set — Baseline vs M4 (Mean)

In [13]:
tabel_12 = build_table(df_val_mean, "Baseline", "M4", stat_type="Mean")
tabel_12.to_csv(os.path.join(OUTPUT_DIR, "tabel_12_val_mean_baseline_vs_m4.csv"), index=False)
display(tabel_12)

,Metrik,Baseline Mean,M4 Mean,Selisih (M4 - Baseline),Nilai P (p-value),Signifikansi (alpha=0.05)
0,CMD-F1,0.3391,0.4395,0.1005,0.0002,Signifikan. M4 lebih baik
1,ARG-F1,0.1855,0.6817,0.4962,0.0002,Signifikan. M4 lebih baik
2,T2C-F1,0.6873,0.9378,0.2505,0.0002,Signifikan. M4 lebih baik
3,Chamfer Distance,11.9877,1524.6433,1512.6556,0.5003,Tidak signifikan
4,Invalid Ratio,0.4664,0.1184,-0.3481,0.0002,Signifikan. M4 lebih baik
